# Ranking the test patients with MWFAS

This notebook:
1. **Loads** the test data (`test/X_test/clinical_test.csv`, `test/X_test/molecular_test.csv`).
2. **Applies the same transformations** as `00_data_processing.ipynb`: drops high-variability molecular columns, classifies cytogenetics, adds DANGER (VAF × DEPTH) and aggregates mutations per patient.
3. **Builds every unique pair** of test patients (pair (i, j) is the same as (j, i), so each pair appears once).
4. **Imputes missing values** with the imputer fitted on the training pairs (`imputer/imputer.pkl`).
5. **Predicts** P(patient i dies before patient j) for every pair with the calibrated logistic regression.
6. **Ranks** all patients by solving a Minimum Weight Feedback Arc Set on the resulting tournament graph (`src/mwfas.py`) and exports the ranking to `submissions/patient_rankings.csv`.

In [ ]:
import itertools

import joblib
import numpy as np
import pandas as pd

from src.dataprocessor import DataProcessor

# =============================================================================
# CONFIGURATION
# =============================================================================
TEST_CLINICAL_PATH = "test/X_test/clinical_test.csv"
TEST_MOLECULAR_PATH = "test/X_test/molecular_test.csv"
IMPUTER_PATH = "imputer/imputer.pkl"

# Columns to remove from molecular data (same as in 00_data_processing.ipynb)
MOL_COLS_TO_REMOVE = ["START", "END", "REF", "ALT", "PROTEIN_CHANGE"]

# Final columns to keep (from treated_data.csv, excluding OS_STATUS and OS_YEARS which are targets)

COLS_FOR_FEATURES = pd.read_csv("data/treated_data.csv", nrows=0).columns.drop(["OS_STATUS", "OS_YEARS"]).tolist()

# Same cytogenetics rules as in training
classify_cytogenetics = DataProcessor.classify_cytogenetics

# =============================================================================
# STEP 1: LOAD TEST DATA
# =============================================================================
print("="*60)
print("STEP 1: Loading test data")
print("="*60)

data_test_mol = pd.read_csv(TEST_MOLECULAR_PATH)
data_test_cli = pd.read_csv(TEST_CLINICAL_PATH)

print(f"Molecular test data shape: {data_test_mol.shape}")
print(f"Clinical test data shape: {data_test_cli.shape}")

# Track patient counts
clinical_patients = set(data_test_cli['ID'].unique())
molecular_patients = set(data_test_mol['ID'].unique())
print(f"Unique patients in clinical: {len(clinical_patients)}")
print(f"Unique patients in molecular: {len(molecular_patients)}")
print(f"Patients in clinical but NOT in molecular: {len(clinical_patients - molecular_patients)}")

# =============================================================================
# STEP 2: DATA TREATMENT (same as in 00_data_processing.ipynb)
# =============================================================================
print("\n" + "="*60)
print("STEP 2: Data treatment")
print("="*60)

# 2.1 Remove high variability columns from molecular data
data_test_mol = data_test_mol.drop(columns=MOL_COLS_TO_REMOVE, errors='ignore')
print(f"After removing high variability columns: {data_test_mol.shape}")

# 2.2 Apply cytogenetics classification to clinical data
if "CYTOGENETICS" in data_test_cli.columns:
    data_test_cli["CYTO_CATEGORY"] = data_test_cli["CYTOGENETICS"].apply(classify_cytogenetics)
    data_test_cli = data_test_cli.drop(columns=["CYTOGENETICS"])
    
    # Create dummy variables for cytogenetics
    cyto_dummies = pd.get_dummies(data_test_cli["CYTO_CATEGORY"], prefix="CYTO")
    data_test_cli = pd.concat([data_test_cli, cyto_dummies], axis=1)
    data_test_cli = data_test_cli.drop(columns=["CYTO_CATEGORY"])
    print("Applied cytogenetics classification and created dummies")

# 2.3 Add DANGER variable (VAF * DEPTH)
data_test_mol["DANGER"] = data_test_mol["VAF"] * data_test_mol["DEPTH"]
print("Added DANGER variable")

# 2.4 Aggregate genes - create dummies and group by patient ID
merged_data = data_test_mol[["ID", "CHR", "GENE", "DANGER", "EFFECT"]].copy()
gene_dummies = pd.get_dummies(merged_data, columns=['GENE', 'CHR', 'EFFECT'])

gene_aggregated = gene_dummies.groupby(['ID']).agg({
    'DANGER': 'sum',
    **{col: 'max' for col in gene_dummies.columns if col not in ['ID', 'DANGER']}
}).reset_index()

print(f"After gene aggregation: {gene_aggregated.shape}")

# 2.5 Merge with clinical data - USE OUTER JOIN TO KEEP ALL PATIENTS
# Start from clinical data to ensure we keep all patients
data_test_processed = data_test_cli.merge(gene_aggregated, on="ID", how="left")

# Fill NaN for patients without molecular data (DANGER=0, all gene dummies=0)
molecular_cols = [col for col in gene_aggregated.columns if col != 'ID']
for col in molecular_cols:
    if col in data_test_processed.columns:
        data_test_processed[col] = data_test_processed[col].fillna(0)

print(f"After merging with clinical data: {data_test_processed.shape}")
print(f"Patients after merge: {len(data_test_processed)}")

# =============================================================================
# STEP 3: ALIGN COLUMNS WITH TRAINING DATA
# =============================================================================
print("\n" + "="*60)
print("STEP 3: Aligning columns with training data")
print("="*60)

# Get all expected columns (excluding ID for now)
expected_cols = [col for col in COLS_FOR_FEATURES if col != 'ID']

# Add missing columns with zeros
for col in expected_cols:
    if col not in data_test_processed.columns:
        data_test_processed[col] = 0
        
# Keep only the expected columns (plus ID)
data_test_final = data_test_processed[COLS_FOR_FEATURES].copy()
print(f"Final processed data shape: {data_test_final.shape}")
print(f"Number of patients preserved: {len(data_test_final)}")

# =============================================================================
# STEP 4: GENERATE ALL UNIQUE PAIRS
# =============================================================================
print("\n" + "="*60)
print("STEP 4: Generating all unique pairs (i,j) where i < j")
print("="*60)

n_patients = len(data_test_final)
print(f"Number of patients: {n_patients}")

# Get all unique pairs using combinations (no duplicates: pair (1,2) is same as (2,1))
X_features = data_test_final.drop(columns=['ID'])
patient_ids = data_test_final['ID'].values

# Generate all unique pairs
indices = list(itertools.combinations(range(n_patients), 2))
print(f"Total unique pairs to generate: {len(indices)}")

X_pairs = []
pair_ids = []

for i, j in indices:
    xi = X_features.iloc[i].values
    xj = X_features.iloc[j].values
    
    # Concatenate features of both patients
    feat = np.concatenate([xi, xj])
    X_pairs.append(feat)
    
    # Store patient IDs
    pair_ids.append((patient_ids[i], patient_ids[j]))

# Create DataFrame
n_features = len(X_pairs[0])
feature_names = [f'feat_{i}' for i in range(n_features)]

X_pairs_df = pd.DataFrame(X_pairs, columns=feature_names)
X_pairs_df.insert(0, 'pair_ids', pair_ids)

print(f"Pairs DataFrame shape: {X_pairs_df.shape}")
print(f"Features per pair: {n_features}")

# =============================================================================
# STEP 5: APPLY IMPUTER FOR MISSING VALUES
# =============================================================================
print("\n" + "="*60)
print("STEP 5: Applying imputer for missing values")
print("="*60)

# Load the saved imputer
imputer = joblib.load(IMPUTER_PATH)
print(f"Loaded imputer from: {IMPUTER_PATH}")

# Check missing values before imputation
print(f"Missing values before imputation: {X_pairs_df.drop(columns=['pair_ids']).isnull().sum().sum()}")

# Separate pair_ids and features
pair_ids_col = X_pairs_df['pair_ids']
X_features_only = X_pairs_df.drop(columns=['pair_ids','feat_40'])

# Apply imputer (transform only, already fitted on training data)
X_imputed = pd.DataFrame(
    imputer.transform(X_features_only),
    columns=X_features_only.columns,
    index=X_features_only.index
)

# Reconstruct DataFrame with pair_ids
X_pairs_imputed = X_imputed.copy()
X_pairs_imputed.insert(0, 'pair_ids', pair_ids_col.values)

print(f"Missing values after imputation: {X_pairs_imputed.drop(columns=['pair_ids']).isnull().sum().sum()}")

# =============================================================================
# FINAL RESULT
# =============================================================================
print("\n" + "="*60)
print("FINAL RESULT")
print("="*60)
print(f"Final pairs DataFrame shape: {X_pairs_imputed.shape}")
print(f"Number of pairs: {len(X_pairs_imputed)}")
print(f"Features per pair: {X_pairs_imputed.shape[1] - 1}")  # -1 for pair_ids column
print(f"All {n_patients} patients preserved")

# Display first few rows
print("\nFirst 5 rows:")
X_pairs_imputed.head()

STEP 1: Loading test data
Molecular test data shape: (3089, 11)
Clinical test data shape: (1193, 9)
Unique patients in clinical: 1193
Unique patients in molecular: 1054
Patients in clinical but NOT in molecular: 139

STEP 2: Data treatment
After removing high variability columns: (3089, 6)
Applied cytogenetics classification and created dummies
Added DANGER variable
After gene aggregation: (1054, 115)
After merging with clinical data: (1193, 125)
Patients after merge: 1193

STEP 3: Aligning columns with training data
Final processed data shape: (1193, 42)
Number of patients preserved: 1193

STEP 4: Generating all unique pairs (i,j) where i < j
Number of patients: 1193
Total unique pairs to generate: 711028


/tmp/ipykernel_85579/1666868892.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  }).reset_index()
/tmp/ipykernel_85579/1666868892.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_test_processed[col] = 0


Pairs DataFrame shape: (711028, 83)
Features per pair: 82

STEP 5: Applying imputer for missing values
Loaded imputer from: imputer/imputer.pkl
Missing values before imputation: 406472
Missing values after imputation: 0

FINAL RESULT
Final pairs DataFrame shape: (711028, 82)
Number of pairs: 711028
Features per pair: 81
All 1193 patients preserved

First 5 rows:


,pair_ids,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,...,feat_72,feat_73,feat_74,feat_75,feat_76,feat_77,feat_78,feat_79,feat_80,feat_81
0,"(KYW1, KYW2)",48.0,7.6,376.608,68.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,"(KYW1, KYW3)",48.0,7.6,376.608,68.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,"(KYW1, KYW4)",48.0,7.6,376.608,68.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,"(KYW1, KYW5)",48.0,7.6,376.608,68.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,"(KYW1, KYW6)",48.0,7.6,376.608,68.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [2]:
# =============================================================================
# STEP 6: LOAD MODEL AND CREATE PROBABILITY MATRIX
# =============================================================================
print("="*60)
print("STEP 6: Loading Logistic Regression model and predicting probabilities")
print("="*60)

MODEL_PATH = "model_calibrated/logistic_regression_tuned_calibrated.pkl"

# Load the trained logistic regression model
model = joblib.load(MODEL_PATH)
print(f"Loaded model from: {MODEL_PATH}")

# Get features for prediction (exclude pair_ids)
X_for_prediction = X_pairs_imputed.drop(columns=['pair_ids'])

# Predict probabilities (probability of class 1 = patient i survives less than patient j)
probabilities = model.predict_proba(X_for_prediction)[:, 1]
print(f"Predicted probabilities for {len(probabilities)} pairs")

# =============================================================================
# STEP 7: CREATE 1193x1193 PROBABILITY MATRIX
# =============================================================================
print("\n" + "="*60)
print("STEP 7: Creating probability matrix")
print("="*60)

# Create a mapping from patient ID to index
unique_patient_ids = data_test_final['ID'].values
n_patients = len(unique_patient_ids)
id_to_idx = {pid: idx for idx, pid in enumerate(unique_patient_ids)}

print(f"Number of patients: {n_patients}")

# Initialize the probability matrix with NaN on diagonal (patient vs itself)
prob_matrix = np.full((n_patients, n_patients), np.nan)

# Fill the matrix
# pair_ids contains tuples (patient_i_id, patient_j_id)
# probabilities[k] = P(patient_i survives less than patient_j) for pair k
for k, (id_i, id_j) in enumerate(pair_ids):
    i = id_to_idx[id_i]
    j = id_to_idx[id_j]
    
    # P(i < j) goes in position [i, j]
    prob_matrix[i, j] = probabilities[k]
    
    # P(j < i) = 1 - P(i < j) goes in position [j, i]
    prob_matrix[j, i] = 1 - probabilities[k]

print(f"Probability matrix shape: {prob_matrix.shape}")
print(f"Non-NaN values (should be {n_patients * (n_patients - 1)}): {np.sum(~np.isnan(prob_matrix))}")

# Display a sample of the matrix
print("\nSample of probability matrix (first 5x5):")
print(prob_matrix[:5, :5])

# Create a DataFrame for better visualization with patient IDs
prob_matrix_df = pd.DataFrame(
    prob_matrix, 
    index=unique_patient_ids, 
    columns=unique_patient_ids
)
print("\nProbability matrix as DataFrame (first 5 patients):")
prob_matrix_df.iloc[:5, :5]

STEP 6: Loading Logistic Regression model and predicting probabilities
Loaded model from: model_calibrated/logistic_regression_tuned_calibrated.pkl
Predicted probabilities for 711028 pairs

STEP 7: Creating probability matrix
Number of patients: 1193
Probability matrix shape: (1193, 1193)
Non-NaN values (should be 1422056): 1422056

Sample of probability matrix (first 5x5):
[[       nan 0.1063083  0.90988704 0.90988704 0.90988704]
 [0.8936917         nan 0.9098867  0.90988704 0.90988704]
 [0.09011296 0.0901133         nan 0.90988704 0.10124106]
 [0.09011296 0.09011296 0.09011296        nan 0.09061709]
 [0.09011296 0.09011296 0.89875894 0.90938291        nan]]

Probability matrix as DataFrame (first 5 patients):


,KYW1,KYW2,KYW3,KYW4,KYW5
KYW1,NaN,0.106308,0.909887,0.909887,0.909887
KYW2,0.893692,NaN,0.909887,0.909887,0.909887
KYW3,0.090113,0.090113,NaN,0.909887,0.101241
KYW4,0.090113,0.090113,0.090113,NaN,0.090617
KYW5,0.090113,0.090113,0.898759,0.909383,NaN


In [3]:
# =============================================================================
# STEP 8: COMPUTE EDGE WEIGHTS USING THE FORMULA
# =============================================================================
print("="*60)
print("STEP 8: Computing edge weights γ_ij")
print("="*60)

# Define the weight function: γ_ij = |2h(x_i,x_j) - 1| * 𝟙(h(x_i,x_j) > 0.5)
def compute_gamma(h_matrix):
    """
    Compute edge weights from probability matrix.
    γ_ij = |2h(x_i,x_j) - 1| * 𝟙(h(x_i,x_j) > 0.5)
    
    Args:
        h_matrix: Probability matrix where h[i,j] = P(i survives less than j)
    
    Returns:
        gamma_matrix: Edge weights matrix with NaN on diagonal
    """
    gamma = np.abs(2 * h_matrix - 1) * (h_matrix > 0.5)
    # Keep diagonal as NaN
    np.fill_diagonal(gamma, np.nan)
    return gamma

# Compute gamma matrix from probabilities
gamma_matrix = compute_gamma(prob_matrix)

print(f"Gamma matrix shape: {gamma_matrix.shape}")
print(f"Non-NaN values: {np.sum(~np.isnan(gamma_matrix))}")
print(f"Non-zero edge weights: {np.sum(gamma_matrix[~np.isnan(gamma_matrix)] > 0)}")

# Display a sample
print("\nSample of gamma matrix (first 5x5):")
print(gamma_matrix[:5, :5])

# =============================================================================
# STEP 9: CREATE EDGE LIST WITH GAMMA WEIGHTS FOR MWFAS
# =============================================================================
print("\n" + "="*60)
print("STEP 9: Creating edge list with gamma weights for MWFAS")
print("="*60)

edges = []
edge_id = 0

for i in range(n_patients):
    for j in range(n_patients):
        if i != j and not np.isnan(gamma_matrix[i, j]):
            # Only add edges with non-zero weight
            if gamma_matrix[i, j] > 0:
                edges.append((i, j, gamma_matrix[i, j], edge_id))
                edge_id += 1

print(f"Created {len(edges)} edges with non-zero weights for the graph")
print(f"Total weight of all edges: {sum(e[2] for e in edges):.6f}")

# =============================================================================
# STEP 10: COMPUTE RANKINGS USING MWFAS
# =============================================================================
print("\n" + "="*60)
print("STEP 10: Computing rankings using MWFAS algorithm")
print("="*60)

from src.mwfas import compute_vertex_rankings, initialize_graph

# Initialize graph structure for MWFAS
graph, weights = initialize_graph(edges)
print(f"Graph initialized with {len(graph)} vertices")

# Compute rankings
rankings = compute_vertex_rankings(graph, weights, n_patients)
print(f"Rankings computed for {len(rankings)} patients")

# =============================================================================
# STEP 11: EXPORT RANKINGS TO CSV
# =============================================================================
print("\n" + "="*60)
print("STEP 11: Exporting rankings to CSV")
print("="*60)

import os

# Create the submissions directory if it doesn't exist
submissions_dir = "submissions"
os.makedirs(submissions_dir, exist_ok=True)

# Create DataFrame with ID and rank
results_df = pd.DataFrame({
    'ID': unique_patient_ids,
    'rank': rankings
})

# Sort by rank (descending - higher rank = better survival)
results_df = results_df.sort_values('rank', ascending=False).reset_index(drop=True)

# Export to CSV
output_path = os.path.join(submissions_dir, "patient_rankings.csv")
results_df.to_csv(output_path, index=False)

print(f"Rankings exported to: {output_path}")
print(f"  - Total patients: {len(results_df)}")
print(f"  - Rank range: {results_df['rank'].min()} to {results_df['rank'].max()}")

# Display first 10 rows
print("\nTop 10 patients by rank:")
results_df.head(10)

STEP 8: Computing edge weights γ_ij
Gamma matrix shape: (1193, 1193)
Non-NaN values: 1422056
Non-zero edge weights: 711028

Sample of gamma matrix (first 5x5):
[[       nan 0.         0.81977408 0.81977408 0.81977408]
 [0.78738339        nan 0.81977341 0.81977408 0.81977408]
 [0.         0.                nan 0.81977408 0.        ]
 [0.         0.         0.                nan 0.        ]
 [0.         0.         0.79751787 0.81876581        nan]]

STEP 9: Creating edge list with gamma weights for MWFAS
Created 711028 edges with non-zero weights for the graph
Total weight of all edges: 569484.134904

STEP 10: Computing rankings using MWFAS algorithm
Graph initialized with 1192 vertices
Rankings computed for 1193 patients

STEP 11: Exporting rankings to CSV
Rankings exported to: submissions/patient_rankings.csv
  - Total patients: 1193
  - Rank range: 0 to 1192

Top 10 patients by rank:


,ID,rank
0,KYW483,1192
1,KYW1120,1191
2,KYW1090,1190
3,KYW1017,1189
4,KYW910,1188
5,KYW866,1187
6,KYW1099,1186
7,KYW891,1185
8,KYW1075,1184
9,KYW927,1183
